In [ ]:
import time
import uuid
import pickle 

from mne_lsl.lsl import (
    StreamInfo, StreamInlet, StreamOutlet, local_clock, resolve_streams
)
from pylsl import resolve_stream
import mne
import os,pandas as pd
import numpy as np
import pathlib

from live_helpers import *
import logging

# For interactive plots
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'qt')

# Define paths: 
current_path = pathlib.Path().absolute()  
recording_path = current_path / 'Recordings'


In [ ]:
# Discover streams, get info
streams = resolve_streams()
print (streams[:])
inlet = StreamInlet(streams[0])
inlet.open_stream()
sinfo = inlet.get_sinfo()  # retrieve stream information with all properties
ch_names = sinfo.get_channel_names()
types = sinfo.get_channel_types()
sinfo.get_channel_units()
sinfo.get_channel_info()
Info = mne.create_info (sinfo.n_channels,sinfo.sfreq,'eeg')

# Ensure the channel names are in the expected format (0 to 66)
original_channel_names = [f'{i}' for i in range(67)]  # Adjust if necessary

# New channel names accroding to the system
new_channel_names = [
    "Fp1", "Fz", "F3", "F7", "FT9", "FC5", "FC1", "C3", "T7",
    "CP5", "CP1", "Pz", "P3", "P7", "TP9", "O1", "Oz", "O2", "TP10",
    "P8", "P4", "CP2", "CP6", "T8", "C4", "Cz", "FC2", "FC6", "FT10",
    "F8", "F4", "Fp2", "AF7", "AF3", "AFz", "F1", "F5", "FT7", "FC3",
    "C1", "C5", "TP7", "CP3", "P1", "P5", "PO7", "PO3", "POz", "PO4",
    "PO8", "P6", "P2", "CPz", "CP4", "TP8", "C6", "C2", "FC4", "FT8",
    "F6", "AF8", "AF4", "F2", "Iz", "ACC_X", "ACC_Y", "ACC_Z"
]

# Check if the original file has extra channels or is missing any, and adjust accordingly
#assert len(Raw.ch_names) >= len(original_channel_names), "The raw file has fewer channels than expected."

# Mapping from original to new names
channel_mapping = dict(zip(original_channel_names, new_channel_names))
montage = mne.channels.read_custom_montage((f"{current_path}\Montages\CACS-64_REF.bvef"), head_size=0.095, coord_frame=None) 

# Create output stream
sdinfo = StreamInfo(
    name="Prediction_stream",
    stype="Markers",
    n_channels=1,
    sfreq=0,
    dtype="string",
    source_id=uuid.uuid4().hex[:6],
)
info = StreamInfo(name='Prediction_stream', stype='Markers', n_channels=1, sfreq=0, dtype='string', source_id='my_source_id')
outlet = StreamOutlet(info)



In [ ]:
# Load model and parameters
fname = 'Gilad_3rd_arm'+'_ts_model_281025'
path_fname = current_path /'Models'/ fname
#%% Load the selected model
#read the pickle file   
picklefile = open(path_fname, 'rb')
#unpickle the dataframe
trained_clf = pickle.load(picklefile)
#close file
picklefile.close()

fname = 'mean'
path_fname = current_path /'Models'/ fname
#create a pickle file
picklefile = open(path_fname, 'rb')
#pickle the dictionary and write it to file
mean = pickle.load(picklefile)
#close the file

picklefile.close()

fname = 'params_dict-281025'
path_fname = current_path /'Models'/ fname
#create a pickle file
picklefile = open(path_fname, 'rb')
#pickle the dictionary and write it to file
params_dict = pickle.load(picklefile)
#close the file

picklefile.close()

In [ ]:
trained_clf_live.classes_

In [ ]:
# Load model and parameters
PerformCsd=params_dict['PerformCsd']
LowPass, HighPass, filter_method = params_dict['LowPass'],params_dict['HighPass'],params_dict['filter_method']
curr_elecs_in_epochs_set=set(Raw.info['ch_names'])
elecs_to_remove=params_dict['bad_electrodes']
picks = params_dict['Electorde_Group']
scale = np.array([1e-6])
# --- Noise rejection thresholds (in microvolts) ---
MAX_ABS_UV   = 100.0   # hard amplitude cap ±150 µV
MIN_STD_UV   = 1.0     # flatline if std < 1 µV
MAX_STD_UV   = 50.0    # EMG burst if std > 50 µV
MAX_BAD_FRAC = 0.20    # reject whole chunk if >20% channels are bad
UV2V = 1e-6


In [ ]:
# give a bit of time to the documentation build after the execution of the last cell
inlet.flush()
time.sleep(0.2)
concat_data = np.empty((0, 67))  # Initialize with zero samples but correct channel count
storage_concat_data = np.empty((0, 67))  # Initialize with zero samples but correct channel count
#assert inlet.samples_available == 1
# Initialize an empty Annotations object
stream_annotations = mne.Annotations(onset=[], duration=[], description=[])

duration = 780
predictions =[]
predictions_proba = []
max_samples = 55
start_time = time.time()
while time.time() - start_time < duration:
    while(inlet.samples_available<55):
        time.sleep(0.001)
    data,ts = inlet.pull_chunk(max_samples=max_samples)
    if ((concat_data.shape)[0]>6000):
        concat_data = concat_data[-5000:]  # Reset 'concat_data' after a while for preprocessing
    concat_data = np.concatenate((concat_data, data), axis=0)
    #For debuging
    now = local_clock()
    print(f"Timestamp of the acquired data: {ts[max_samples-1]}")
    print(f"Current time: {now}")
    print(f"Delta: {now - ts[max_samples-1]} seconds")
    
    # Temporary suppress logging output
    original_log_level = logging.getLogger('mne').getEffectiveLevel()
    logging.getLogger('mne').setLevel(logging.ERROR)  # Suppress messages below ERROR level
    Raw=mne.io.RawArray((concat_data * scale).T,Info)
    # Calculate the time of the last sample
    last_sample_time = time.time() - start_time

    # Rename channels
    Raw.rename_channels(channel_mapping)
    #mne.rename_channels(Raw.info, {'F9' : 'FT9','P9' : 'TP9','P10' : 'TP10','F10' : 'FT10','AF1' : 'AF7' }, allow_duplicates=False, verbose=None)
    Raw.drop_channels(['ACC_X','ACC_Y','ACC_Z']) ## Drop non eeg channels

    Raw.set_montage(montage, match_case=True, match_alias=False, on_missing='raise', verbose=None)
    curr_elecs_in_epochs_set=set(Raw.info['ch_names'])
    elecs_to_remove=params_dict['bad_electrodes']
    elecs_to_drop=curr_elecs_in_epochs_set.intersection(elecs_to_remove)
    if len(elecs_to_drop)>0: 
        Raw.drop_channels(list(elecs_to_drop))
    
    Raw.drop_channels(Raw.info['bads'])
    if (params_dict['PerformAvgRef']):
        Raw.set_eeg_reference(ref_channels="average")
    unfiltered_Raw = Raw.copy()
    notched_Raw = unfiltered_Raw.filter(1, 100, method=filter_method, phase='forward', pad=0)  
    notched_Raw.notch_filter(50, method=filter_method, phase='forward') 
    eeg_chunk_volts = notched_Raw.get_data()[:,-1000:]
    accept, reason = gate_chunk(eeg_chunk_volts)
    if accept:
        if PerformCsd:
            notched_Raw = mne.preprocessing.compute_current_source_density(notched_Raw) # Perform current source density
        Raw_Filtered = notched_Raw.filter(LowPass, HighPass, method=filter_method, iir_params = dict(order=4, ftype='butter'),phase='forward',pad=0)
        Raw_Filtered.pick(picks)
        Output_data= (Raw_Filtered.get_data())[:, -1000:]
        data_for_classification = Output_data[np.newaxis, :, :]

        # Restore original logging level
        logging.getLogger('mne').setLevel(original_log_level)
        prediction_decision_function = trained_clf.decision_function(data_for_classification)
        prediction_proba = trained_clf.predict_proba(data_for_classification)
        prediction=trained_clf.predict(data_for_classification)
    else:
        print(reason)
        prediction = ['Noise']
    if len(predictions)>1:
        if (predictions[-1:][0] != prediction):
            # Create an annotation at the last sample time
            new_annotations = mne.Annotations(onset=[last_sample_time], duration=[0], description=prediction)
            # Add the annotation to the Raw object
            stream_annotations += new_annotations  # This merges the new annotation with the existing ones
            outlet.push_sample([prediction[0]])

    print(prediction)
    predictions.append(prediction)
    #predictions_proba.append(prediction_proba)
    # Push processed data chunk for classification
    #assert inlet.samples_available == 0


Comparison and analysis with true annotations

Raw_for_analysis.set_annotations(stream_annotations)
Raw_for_analysis.plot()